# 65 - Production-Independent Candidate Pooling

The one direct ranking test run so far (Section~\ref{subsec:goi_vs_ours_ranking}) found production's own ranking beating an independently cross-validated ranker at every $k$, but the gold standard it was scored against was partly pooled from production's own output, giving production a structural advantage. This notebook builds a genuinely production-independent candidate pool instead: every candidate here comes exclusively from the random, query-independent 300K-company sample (`source == "new"` in `combined_pool.parquet`), companies production was never asked to rank for any of these queries, so there is no way for production's own output to have influenced which candidates end up here.

Scope: the 19 deep-coverage queries (5-query pilot + 14 headline queries), the same set used throughout this thesis's other rigorous comparisons, keeping the LLM-judging cost in the next notebook manageable (roughly (19/101) of the original $22 full-pool judging cost, so a few dollars). For each query, candidates are pooled from four independent retrieval channels (BM25, MiniLM, GTE-large, Linq-Embed-Mistral), searched *only* over the `source == "new"` companies so the ranking is never diluted by the already-curated existing corpus, then trust-filtered the same way as the original gold standard (Section~\ref{subsec:pooled_gold_standard}) before being handed off to notebook 66 for LLM judging.

In [ ]:
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize

OUTPUT_DIR = Path("result/65_production_independent_pooling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEEP_QUERY_IDS = [1, 2, 3, 4, 5, 11, 12, 15, 34, 14, 27, 66, 72, 99, 101, 56, 82, 91, 92]
TOP_N = 20  # candidates pulled per channel per query, before dedup -- matches notebook 34's original pooling depth

combined = pd.read_parquet("result/44_build_scaled_corpus/combined_pool.parquet")
new_mask = (combined["source"] == "new").values
print(f"[Load] Combined pool: {len(combined):,} companies")
print(f"[Load] source=='new' (random sample, production-independent): {new_mask.sum():,} companies")

new_domains = combined.loc[new_mask, "domain"].reset_index(drop=True)

train_queries = json.load(open("result/08_llm_relevance_judge/train_queries.json"))
held_out_queries = json.load(open("result/08_llm_relevance_judge/held_out_queries.json"))
query_lookup = {item["query_id"]: item["query"] for item in train_queries + held_out_queries}
deep_queries = [(qid, query_lookup[qid]) for qid in DEEP_QUERY_IDS]
print(f"[Load] Deep-coverage queries: {len(deep_queries)}")

In [ ]:
tokens_path = Path("result/59_bm25_tiered_index/tokenized_corpus.json")
if tokens_path.exists():
    print("[BM25] Reusing notebook 59's already-tokenized corpus (same row order as combined_pool.parquet)")
    tokenized_corpus = json.load(open(tokens_path))
else:
    print("[BM25] No cached tokenization found -- tokenizing rich_text now (this will take a few minutes)")
    tokenized_corpus = [
        word_tokenize(text.lower()) if isinstance(text, str) and text else []
        for text in combined["rich_text"].tolist()
    ]

new_tokenized = [tokenized_corpus[i] for i in np.where(new_mask)[0]]
print(f"[BM25] Building index over {len(new_tokenized):,} source=='new' companies only...")
t0 = time.time()
bm25_new = BM25Okapi(new_tokenized)
print(f"[BM25] Index built in {time.time()-t0:.1f}s")

In [ ]:
from sentence_transformers import SentenceTransformer
import os
import torch

os.environ["HF_HUB_OFFLINE"] = "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
query_texts = [qtext for _, qtext in deep_queries]

print("[Encode] MiniLM...")
try:
    minilm_model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE, local_files_only=True)
except Exception:
    minilm_model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
minilm_query_embs = minilm_model.encode(query_texts, normalize_embeddings=False, convert_to_numpy=True).astype("float32")
del minilm_model

print("[Encode] GTE-large...")
try:
    gte_model = SentenceTransformer("thenlper/gte-large", device=DEVICE, local_files_only=True)
except Exception:
    gte_model = SentenceTransformer("thenlper/gte-large", device=DEVICE)
gte_query_embs = gte_model.encode(query_texts, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
del gte_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("[Encode] Linq-Embed-Mistral...")
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

TASK_INSTRUCTION = "Given a search query describing a type of company, retrieve relevant company profiles"
query_prefix = f"Instruct: {TASK_INSTRUCTION}\nQuery: "
REPO = "Linq-AI-Research/Linq-Embed-Mistral"
try:
    linq_tokenizer = AutoTokenizer.from_pretrained(REPO, local_files_only=True)
    linq_model = AutoModel.from_pretrained(REPO, torch_dtype=torch.float16, device_map=DEVICE, local_files_only=True)
except Exception:
    linq_tokenizer = AutoTokenizer.from_pretrained(REPO)
    linq_model = AutoModel.from_pretrained(REPO, torch_dtype=torch.float16, device_map=DEVICE)


def last_token_pool(last_hidden_states, attention_mask):
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


@torch.no_grad()
def encode_linq(texts):
    batch_dict = linq_tokenizer(texts, max_length=512, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
    outputs = linq_model(**batch_dict)
    embs = last_token_pool(outputs.last_hidden_state, batch_dict["attention_mask"])
    embs = F.normalize(embs, p=2, dim=1)
    return embs.cpu().float().numpy()


linq_query_embs = encode_linq([query_prefix + q for q in query_texts]).astype("float32")
del linq_model
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print(f"[Encode] Done. minilm={minilm_query_embs.shape}, gte={gte_query_embs.shape}, linq={linq_query_embs.shape}")

In [ ]:
minilm_embs_new = np.load("result/45_encode_minilm_scaled/company_embeddings.npy", mmap_mode="r")[new_mask].astype("float32")
gte_embs_new = np.load("result/46_encode_gte_scaled/company_embeddings.npy", mmap_mode="r")[new_mask].astype("float32")
linq_embs_new = np.load("result/47_encode_linq_mistral_scaled/company_embeddings.npy", mmap_mode="r")[new_mask].astype("float32")
print(f"[Load] source=='new' embeddings: minilm={minilm_embs_new.shape}, gte={gte_embs_new.shape}, linq={linq_embs_new.shape}")


def top_n_domains(scores, domains, n, ascending):
    order = np.argsort(scores) if ascending else np.argsort(scores)[::-1]
    return [domains.iloc[i] for i in order[:n]]


domain_sources = {qid: {} for qid, _ in deep_queries}

for i, (qid, qtext) in enumerate(deep_queries):
    # BM25
    q_tokens = word_tokenize(qtext.lower())
    bm25_scores = bm25_new.get_scores(q_tokens)
    for d in top_n_domains(bm25_scores, new_domains, TOP_N, ascending=False):
        domain_sources[qid].setdefault(d, []).append("bm25")

    # MiniLM (L2 distance -- lower is better)
    minilm_dists = np.sum((minilm_embs_new - minilm_query_embs[i]) ** 2, axis=1)
    for d in top_n_domains(minilm_dists, new_domains, TOP_N, ascending=True):
        domain_sources[qid].setdefault(d, []).append("minilm")

    # GTE (cosine/inner product -- higher is better)
    gte_sims = gte_embs_new @ gte_query_embs[i]
    for d in top_n_domains(gte_sims, new_domains, TOP_N, ascending=False):
        domain_sources[qid].setdefault(d, []).append("gte")

    # Linq (cosine/inner product -- higher is better)
    linq_sims = linq_embs_new @ linq_query_embs[i]
    for d in top_n_domains(linq_sims, new_domains, TOP_N, ascending=False):
        domain_sources[qid].setdefault(d, []).append("linq")

    print(f"  query {qid} ({qtext}): {len(domain_sources[qid])} unique pooled candidates")

print("[Pool] Done pooling all 19 queries.")

In [ ]:
meta_cols = ["domain", "name", "organization_type", "organization_size", "country",
             "state", "district", "municipality", "summary", "summary_keywords", "nace_code"]
meta_lookup = combined[meta_cols].drop_duplicates(subset="domain").set_index("domain")

pooled_rows = []
for qid, qtext in deep_queries:
    for domain, sources in domain_sources[qid].items():
        if domain not in meta_lookup.index:
            continue
        row = meta_lookup.loc[domain]
        pooled_rows.append({
            "query_id": qid, "query": qtext, "domain": domain,
            "sources": sources, "n_sources": len(sources),
            "name": row["name"], "organization_type": row["organization_type"],
            "organization_size": row["organization_size"], "country": row["country"],
            "state": row["state"], "district": row["district"], "municipality": row["municipality"],
            "summary": row["summary"], "summary_keywords": row["summary_keywords"], "nace_code": row["nace_code"],
        })

pooled_df = pd.DataFrame(pooled_rows)
print(f"[Pool] {len(pooled_df):,} pooled candidates across {pooled_df['query_id'].nunique()} queries")
print(f"[Pool] Average unique candidates per query: {len(pooled_df) / pooled_df['query_id'].nunique():.1f}")

In [ ]:
import trust_feature as tf

embedder = tf.get_embedder()
template_texts, template_embeddings = tf.load_boilerplate_templates(embedder=embedder)
print(f"[Trust] Loaded {len(template_texts)} boilerplate templates")

verdicts = tf.summary_verdicts_batch(
    pooled_df["name"].tolist(),
    pooled_df["domain"].tolist(),
    pooled_df["summary"].tolist(),
    template_embeddings,
    embedder,
)
pooled_df["summary_trustworthy"] = [ok for ok, _ in verdicts]
pooled_df["summary_reasons"] = ["; ".join(reasons) for _, reasons in verdicts]

n_bad = (~pooled_df["summary_trustworthy"]).sum()
print(f"[Trust] Flagged untrustworthy: {n_bad}/{len(pooled_df)} ({100*n_bad/len(pooled_df):.1f}%)")

out_path = OUTPUT_DIR / "pooled_candidates.json"
pooled_df.to_json(out_path, orient="records", indent=2)
print(f"[Pool] Saved -> {out_path}")
print()
print("Per-query pool size distribution:")
print(pooled_df.groupby("query_id").size().describe())